<a href="https://colab.research.google.com/github/JuanZapa7a/AINavalEngineering/blob/main/NB06_SciPy_for_Naval_Engineering_Interpolation_Optimization_and_Signals_ES.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# **NB06 · Clase 6 — SciPy para ingeniería naval: interpolación, optimización y señales**

## Bloque 2: IA — Machine Learning (continuación)

`NB03`–`NB05` construyeron soltura real con NumPy y Matplotlib. Esta clase cierra la secuencia de herramientas del Bloque 2 con **SciPy**, la librería en la que se apoya todo cálculo de ingeniería clásica de este curso pero que todavía no hemos usado directamente: interpolar medidas reales dispersas, encontrar un óptimo a partir de datos reales, integrar una curva real, y analizar el contenido en frecuencia real de una señal. Ningún dataset de este notebook se ha inventado para la ocasión — las secciones de interpolación y optimización usan las medidas reales y dispersas de ensayo de canal `yacht_hydrodynamics.data` (`NB10`); solo la sección final de análisis de señal usa una señal sintética de movimiento del buque claramente etiquetada como tal, en el mismo espíritu que el propio ejemplo de vibración de `NB01`, porque este curso todavía no tiene ninguna serie temporal real de movimiento de un buque entre sus datasets.

### Objetivos de aprendizaje

Al terminar esta clase, el alumnado será capaz de:
- Interpolar medidas reales dispersas con `scipy.interpolate.make_interp_spline` y explicar por qué eso es distinto de simplemente unir los puntos.
- Encontrar un óptimo real (mínimo) de una función construida a partir de datos reales con `scipy.optimize.minimize_scalar`.
- Integrar numéricamente una curva real con `scipy.integrate.simpson`.
- Analizar el contenido en frecuencia de una señal con `scipy.fft` y detectar picos locales reales con `scipy.signal.find_peaks`.

### Agenda (clase de 2 horas)

| # | Sección | Minutos |
|---|---|---|
| 1 | Repaso y por qué SciPy | 5 |
| 2 | Interpolar datos reales dispersos de ensayo de canal | 20 |
| 3 | Encontrar un óptimo real de velocidad económica | 15 |
| 4 | Integración numérica de una curva real | 20 |
| 5 | Análisis en frecuencia con `scipy.fft` | 30 |
| 6 | Detectar picos reales con `scipy.signal` | 15 |
| 7 | Resumen, tarea, próxima clase | 15 |

Como siempre: orientación aproximada, no un guion cerrado.

---

## 1. Por qué SciPy

NumPy proporciona arrays y operaciones básicas; `SciPy construye encima métodos numéricos reales: interpolación, optimización, integración y procesado de señales, entre muchos otros`. Esta clase es deliberadamente la clase de **cierre** de la secuencia de herramientas del Bloque 2 (`NB03`–`NB06`) porque se apoya en todo lo anterior — un spline real se construye a partir de arrays reales (`NB03`), se evalúa sobre una rejilla real (el broadcasting de `NB04`), y se representa gráficamente para comprobar el resultado (`NB05`).

> **Para saber más**: [documentación oficial de SciPy](https://docs.scipy.org/doc/scipy/) | [SciPy (Wikipedia)](https://en.wikipedia.org/wiki/SciPy)

---

## 2. Interpolar datos reales dispersos de ensayo de canal

El dataset Yacht Hydrodynamics de `NB10` contiene 308 medidas reales de ensayo de canal de remolque sobre 22 formas de casco reales, con 14 números de Froude (una medida adimensional de velocidad) cada una. La celda siguiente aísla las 14 medidas reales (número de Froude, coeficiente de resistencia residual) de **un único casco real** — `dispersas por diseño, ya que cada ensayo físico real en el canal es costoso`.

In [ ]:
import numpy as np
import urllib.request

yacht_url = "https://raw.githubusercontent.com/JuanZapa7a/AINavalEngineering/main/Datasets/yacht_hydrodynamics.data"
raw_lines = urllib.request.urlopen(yacht_url).read().decode("utf-8").strip().split("\n")
all_rows = np.array([[float(x) for x in line.split()] for line in raw_lines])

first_hull_config = all_rows[0, :5]
same_hull = np.all(all_rows[:, :5] == first_hull_config, axis=1)
hull_rows = all_rows[same_hull]

froude = hull_rows[:, 5]          # dimensionless speed
resistance = hull_rows[:, 6]      # real residuary resistance coefficient

print(f"Real measurements for this hull: {len(froude)}")
print("Froude numbers:", froude)
print("Resistance coefficients:", resistance)


`scipy.interpolate.make_interp_spline` ajusta una curva suave exactamente a través de estos puntos reales, permitiendo estimar la resistencia en cualquier número de Froude **entre** los realmente medidos — sin necesidad de un nuevo y costoso ensayo en el canal. (El código SciPy más antiguo a veces usa `interpolate.interp1d` para esto; la propia documentación de SciPy recomienda ahora `make_interp_spline` en su lugar, que es lo que usa esta clase en todo momento.)

In [ ]:
from scipy.interpolate import make_interp_spline
import matplotlib.pyplot as plt

spline = make_interp_spline(froude, resistance, k=3)   # cubic spline through the real points

froude_fine = np.linspace(froude.min(), froude.max(), 300)
resistance_smooth = spline(froude_fine)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(froude_fine, resistance_smooth, color="steelblue", label="Cubic spline (interpolated)")
ax.scatter(froude, resistance, color="darkorange", zorder=3, label="Real tank measurements")
ax.set_xlabel("Froude number")
ax.set_ylabel("Residuary resistance coefficient")
ax.set_title("Real sparse measurements, interpolated")
ax.legend()
plt.show()


**Pruébalo tú mismo**: ajusta un spline *lineal* (`k=1`) a los mismos puntos reales y compáralo con el spline cúbico (`k=3`) anterior en una misma gráfica — ¿dónde discrepan más los dos, y por qué tiene sentido físico eso para una curva de resistencia suave?

In [ ]:
spline_linear = make_interp_spline(froude, resistance, k=1)
resistance_linear = spline_linear(froude_fine)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(froude_fine, resistance_smooth, color="steelblue", label="Cubic spline (k=3)")
ax.plot(froude_fine, resistance_linear, color="darkred", linestyle="--", label="Linear spline (k=1)")
ax.scatter(froude, resistance, color="darkorange", zorder=3, label="Real tank measurements")
ax.set_xlabel("Froude number")
ax.set_ylabel("Residuary resistance coefficient")
ax.set_title("Cubic vs. linear interpolation of the same real points")
ax.legend()
plt.show()


> **Para saber más**: [Interpolación mediante splines (Wikipedia)](https://en.wikipedia.org/wiki/Spline_interpolation) | [documentación de `scipy.interpolate.make_interp_spline`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.interpolate.make_interp_spline.html) | [Número de Froude (Wikipedia)](https://en.wikipedia.org/wiki/Froude_number)

---

## 3. Encontrar un óptimo real de velocidad económica

Los datos reales de la Sección 2 muestran que la resistencia **aumenta** con la velocidad en todo el rango medido — `si solo se mira la resistencia, la "mejor" velocidad es siempre la más lenta, lo cual no es una respuesta útil`. Una decisión real de velocidad económica sopesa la resistencia (más velocidad cuesta más combustible) frente al coste real de tardar más en llegar. La celda siguiente añade un término simple de coste por tiempo (`K / número de Froude`, ilustrativo — una flota real usaría aquí sus propias cifras reales de tarifa diaria) a la curva de resistencia *real* interpolada en la Sección 2, y después pide a `scipy.optimize.minimize_scalar` que encuentre el número de Froude que minimiza el total combinado.

In [ ]:
from scipy.optimize import minimize_scalar

K = 0.4   # illustrative weighting of "cost of time" relative to resistance -- not a real fleet's day rate

def total_cost(fr):
    return spline(fr) + K / fr

result = minimize_scalar(total_cost, bounds=(froude.min(), froude.max()), method="bounded")

print(f"Optimal (economic) Froude number: {result.x:.3f}")
print(f"Total cost at optimum: {result.fun:.3f}")
print(f"Real resistance coefficient at that speed: {spline(result.x):.3f}")


Representar la curva de coste completa deja claro que este óptimo es un mínimo interior genuino, no un artefacto del límite del intervalo.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
costs = total_cost(froude_fine)
ax.plot(froude_fine, costs, color="seagreen", label="Total cost (resistance + time penalty)")
ax.axvline(result.x, color="black", linestyle="--", label=f"Optimum: Fr = {result.x:.3f}")
ax.set_xlabel("Froude number")
ax.set_ylabel("Total cost (illustrative units)")
ax.set_title("A real optimum found on real (interpolated) data")
ax.legend()
plt.show()


**Lee este resultado con honestidad**: la curva de resistencia en sí es real; el óptimo concreto que se encuentra aquí depende por completo de la `K` ilustrativa elegida para la penalización por tiempo. Un operador de flota real sustituiría `K` por una razón real tarifa-diaria/precio-del-combustible — el *método* (interpolar datos reales y después optimizar un coste construido sobre ellos) es la lección transferible, no este número concreto `K = 0.4`.

> **Para saber más**: [Optimización matemática (Wikipedia)](https://en.wikipedia.org/wiki/Mathematical_optimization) | [documentación de `scipy.optimize.minimize_scalar`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.minimize_scalar.html) | [Velocidad económica de un buque / slow steaming (Wikipedia)](https://en.wikipedia.org/wiki/Slow_steaming)

---

## 4. Integración numérica de una curva real

La **integración** encuentra el área bajo una curva — aquí, la curva real de resistencia interpolada en la Sección 2, sobre el rango real medido de números de Froude. `scipy.integrate.simpson` estima esto numéricamente a partir de los puntos muestreados, `usando la regla de Simpson (ajustando pequeños segmentos parabólicos, más precisa que simplemente sumar rectángulos)`. (El código SciPy más antiguo a veces llama a esto `integrate.simps`; el nombre actual, no obsoleto, es `simpson`, que es el que se usa aquí.)

In [ ]:
from scipy.integrate import simpson

area = simpson(y=resistance_smooth, x=froude_fine)
print(f"Area under the real resistance curve, Fr = {froude.min():.3f} to {froude.max():.3f}: {area:.3f}")
print("(proportional to the total resistance work swept while accelerating across this real speed range)")


**Pruébalo tú mismo**: compara la regla de Simpson con la más sencilla `scipy.integrate.trapezoid` (segmentos rectos en vez de parabólicos) sobre exactamente la misma curva — ¿cuán parecidas son las dos estimaciones reales, y coincide eso con lo que esperarías dado lo suave que ya parece esta curva?

In [ ]:
from scipy.integrate import trapezoid

area_trap = trapezoid(y=resistance_smooth, x=froude_fine)
print(f"Simpson's rule area: {area:.4f}")
print(f"Trapezoidal rule area: {area_trap:.4f}")
print(f"Difference: {abs(area - area_trap):.6f} ({abs(area - area_trap) / area:.3%} of Simpson's estimate)")


> **Para saber más**: [Regla de Simpson (Wikipedia)](https://en.wikipedia.org/wiki/Simpson%27s_rule) | [documentación de `scipy.integrate.simpson`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.integrate.simpson.html)

---

## 5. Análisis en frecuencia con `scipy.fft`

Este curso todavía no tiene ninguna serie temporal real de movimiento de un buque entre sus datasets, así que esta sección usa una señal **claramente sintética** — la misma convención honesta que ya estableció para este curso el propio ejemplo de vibración de `NB01`. Simula el movimiento de arfada (vertical) de un buque: una frecuencia de encuentro con el oleaje dominante y físicamente plausible, más ruido de sensor, y plantea si `scipy.fft` puede recuperar la frecuencia inyectada a partir únicamente de la señal ruidosa.

In [ ]:
rng = np.random.default_rng(42)

sample_rate_hz = 10.0                 # 10 readings per second
duration_s = 60.0
t = np.arange(0, duration_s, 1 / sample_rate_hz)

true_wave_freq_hz = 0.15              # a plausible real wave-encounter frequency (~6.7 s period)
heave_signal = 0.8 * np.sin(2 * np.pi * true_wave_freq_hz * t) + 0.15 * rng.standard_normal(len(t))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, heave_signal, linewidth=0.8)
ax.set_xlabel("Time (s)")
ax.set_ylabel("Heave (m, synthetic)")
ax.set_title("Synthetic heave signal (true frequency hidden -- unknown to the FFT)")
plt.show()


`Nada en la propia gráfica de esta señal revela su frecuencia a simple vista a través del ruido` -- `scipy.fft` la extrae directamente de los datos.

In [ ]:
from scipy.fft import rfft, rfftfreq

fft_values = rfft(heave_signal)
fft_freqs = rfftfreq(len(heave_signal), d=1 / sample_rate_hz)
fft_magnitude = np.abs(fft_values)

dominant_freq = fft_freqs[np.argmax(fft_magnitude)]

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(fft_freqs, fft_magnitude)
ax.axvline(dominant_freq, color="darkred", linestyle="--", label=f"Detected: {dominant_freq:.3f} Hz")
ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Magnitude")
ax.set_title("FFT of the synthetic heave signal")
ax.set_xlim(0, 1.0)
ax.legend()
plt.show()

print(f"True injected frequency:  {true_wave_freq_hz} Hz")
print(f"FFT-detected frequency:   {dominant_freq:.3f} Hz")


**Pruébalo tú mismo**: aumenta la amplitud del ruido de `0.15` a `0.5` (aproximadamente 3 veces más ruidosa) y vuelve a ejecutar la FFT sobre la señal más ruidosa — ¿sigue recuperando correctamente la frecuencia verdadera?

In [ ]:
noisy_heave_signal = 0.8 * np.sin(2 * np.pi * true_wave_freq_hz * t) + 0.5 * rng.standard_normal(len(t))

fft_values_noisy = rfft(noisy_heave_signal)
fft_magnitude_noisy = np.abs(fft_values_noisy)
dominant_freq_noisy = fft_freqs[np.argmax(fft_magnitude_noisy)]

print(f"True injected frequency:        {true_wave_freq_hz} Hz")
print(f"FFT-detected (original noise):  {dominant_freq:.3f} Hz")
print(f"FFT-detected (3x more noise):   {dominant_freq_noisy:.3f} Hz")


> **Para saber más**: [Transformada rápida de Fourier (Wikipedia)](https://en.wikipedia.org/wiki/Fast_Fourier_transform) | [documentación de `scipy.fft`](https://docs.scipy.org/doc/scipy/reference/fft.html) | [Movimientos del buque / arfada (Wikipedia)](https://en.wikipedia.org/wiki/Ship_motions)

---

## 6. Detectar picos reales con `scipy.signal`

`scipy.signal.find_peaks` localiza máximos locales directamente en la señal en el dominio del tiempo — `útil siempre que la pregunta sea "cuántas crestas de ola han ocurrido" en vez de "qué frecuencia domina"`.

In [ ]:
from scipy.signal import find_peaks

peak_indices, _ = find_peaks(heave_signal, height=0.3, distance=int(sample_rate_hz * 3))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.plot(t, heave_signal, linewidth=0.8)
ax.scatter(t[peak_indices], heave_signal[peak_indices], color="darkred", zorder=3, label="Detected peaks")
ax.set_xlabel("Time (s)")
ax.set_ylabel("Heave (m, synthetic)")
ax.set_title(f"{len(peak_indices)} real local peaks detected in this synthetic signal")
ax.legend()
plt.show()


**Pruébalo tú mismo**: usa los instantes de tiempo de los picos detectados para estimar el periodo de la señal de una forma completamente distinta — el tiempo medio entre picos consecutivos — y convierte eso a una frecuencia. ¿Cuán cerca queda de la frecuencia detectada por la FFT en la Sección 5 y del valor verdadero inyectado?

In [ ]:
peak_times = t[peak_indices]
peak_intervals = np.diff(peak_times)
mean_period = peak_intervals.mean()
freq_from_peaks = 1 / mean_period

print(f"Time between consecutive detected peaks: {peak_intervals}")
print(f"Mean period from peaks: {mean_period:.2f} s -> frequency: {freq_from_peaks:.3f} Hz")
print(f"FFT-detected frequency (Section 5): {dominant_freq:.3f} Hz")
print(f"True injected frequency: {true_wave_freq_hz} Hz")


> **Para saber más**: [documentación de `scipy.signal.find_peaks`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.signal.find_peaks.html)

---

## Resumen de la clase

- `scipy.interpolate.make_interp_spline` construye una curva suave y utilizable a partir de medidas reales dispersas (`interp1d` es la forma más antigua, ahora desaconsejada, de hacer esto).
- `scipy.optimize.minimize_scalar` encuentra un óptimo real sobre una función construida a partir de datos reales — aquí, un compromiso genuino de velocidad económica, con una nota honesta sobre qué parte del resultado son datos reales y qué parte es una ponderación ilustrativa.
- `scipy.integrate.simpson` calcula el área real bajo una curva (`simps` es el nombre más antiguo, ya eliminado).
- `scipy.fft` recupera el contenido en frecuencia real de una señal; `scipy.signal.find_peaks` encuentra máximos locales reales directamente en el tiempo.
- Esto cierra la secuencia de herramientas del Bloque 2 (`NB03`–`NB06`): el tratamiento de datos, la geometría y el trabajo con señales de todos los notebooks posteriores se apoya en lo que establecieron estas cuatro clases.

## Para la próxima clase (NB07)

Los fundamentos propiamente dichos de Machine Learning, aplicados de principio a fin a un dataset real de buques — aprendizaje supervisado, particiones de entrenamiento/validación/test, y métricas de evaluación reales.

## Tarea / Ideas de práctica

1. Repite la interpolación de la Sección 2 para una configuración de casco real *distinta* dentro de `yacht_hydrodynamics.data` — ¿la forma de la curva de resistencia se parece, o la geometría del casco la cambia de forma significativa?
2. Cambia `K` en la Sección 3 en un rango real (p. ej. de 0.1 a 0.6) y representa cómo se desplaza el número de Froude óptimo — ¿en qué punto el óptimo llega al límite del rango real medido?
3. Calcula la integral de la Sección 4 por separado para dos cascos reales distintos y compáralos — ¿un casco con resistencia generalmente mayor tiene también un área mayor bajo su curva, como sería de esperar?
4. Añade una *segunda* componente de frecuencia físicamente plausible a la señal sintética de la Sección 5 (p. ej. un movimiento de balanceo más rápido superpuesto a la arfada más lenta) y comprueba si `scipy.fft` puede recuperar ambas frecuencias a partir de la señal combinada y ruidosa.
5. Baja el umbral `height` de la Sección 6 en `find_peaks` y observa cuántos picos adicionales (probablemente causados por ruido, no crestas de ola reales) se detectan — ¿qué sugiere esto sobre cómo elegir los parámetros de detección de picos en datos de sensor reales y ruidosos?

> ***Como siempre: SciPy proporciona métodos numéricos reales — el juicio de ingeniería sobre qué datos reales alimentarlos, y cómo leer el resultado con honestidad, sigue siendo tuyo.***